# Livrable 1 Classification d'images (TouNum)

 

## Guide d'utilisation du notebook

Ce notebook est structuré en **3 parties indépendantes**. Chaque partie peut être exécutée seule (toutes les dépendances et variables sont redéclarées localement).

### Structure

| Partie | Description | Sections |
|  --|    -|   -|
| **Partie 1** | Classification **binaire** (Photo / Not-Photo) | 1.1 sans class weights · 1.2 avec class weights |
| **Partie 2** | Classification **multi-classes à 5 classes** (CNN perso) | 2.1 sans class weights · 2.2 avec class weights |
| **Partie 3** | Classification **multi-classes via Transfer Learning** (VGG19) | 3.1 sans class weights · 3.2 avec class weights |

### Prérequis avant d'exécuter

1. **Configurer les chemins** : Dans chaque section, modifier les variables `DATASET_DIR`, `MODEL_SAVE_DIR` et `LOG_DIR` pour correspondre à votre environnement local.
2. **Structure du dataset** : Le répertoire `DATASET_DIR` doit contenir **un sous-dossier par classe** (ex: `Photo/`, `Dessin/`, `Schema/`, ...). Le split 60/20/20 est géré automatiquement par le code.
3. **Créer les dossiers de sauvegarde** : Les dossiers `MODEL_SAVE_DIR` et `LOG_DIR` doivent exister avant l'exécution (ou seront créés automatiquement).
4. **TensorBoard** : Pour visualiser l'entraînement en temps réel, lancer dans un terminal :
   ```
   tensorboard --logdir <votre LOG_DIR>
   ```
   Puis ouvrir `http://localhost:6006` dans un navigateur.

### Split des données (60 / 20 / 20)

- **60%** entraînement (`train`)
- **20%** validation (utilisée pendant `model.fit` pour suivre les métriques)
- **20%** test final (évaluée **après** l'entraînement, jamais vue par le modèle)

### Early Stopping

Chaque entraînement utilise un `EarlyStopping` sur `val_loss` (patience = 8 epochs) pour éviter le sur-apprentissage. Le meilleur modèle est automatiquement sauvegardé via `ModelCheckpoint`.

### Résultats produits par chaque section

- Courbes `val_accuracy` et `val_loss`
- Évaluation sur le **test set** (loss + accuracy)
- **Matrice de confusion** sur le test set
- Rapport de classification détaillé (precision, recall, F1)
- Analyse qualitative (succès et erreurs du modèle)
- Logs **TensorBoard**


# Contexte du Projet

Dans le cadre du projet TouNum, l'objectif est de concevoir un système automatisé capable de trier et de classifier des flux d'images. Ce module implémente plusieurs stratégies de classification pour répondre aux besoins opérationnels :

- **Classification Binaire** : Discriminer les images de type *Photo* des autres catégories *Not-Photo*.
- **Classification Multi-classes (5 classes)** : Identifier précisément la nature de l'image parmi l'ensemble des catégories disponibles.

## Démarche Méthodologique

1. **Fiabilisation des données** : Détection et suppression des fichiers corrompus.
2. **Pipeline & Optimisation** : Chargement asynchrone (Prefetch) pour maximiser l'utilisation du GPU.
3. **Split 60/20/20** : Séparation rigoureuse en train, validation et test.
4. **Modélisation** : CNN sur mesure + Transfer Learning VGG19.
5. **Gestion du déséquilibre** : Class Weights (avec et sans).
6. **Analyse des Performances** : TensorBoard, courbes, matrices de confusion, analyse qualitative des erreurs.

# Partie 1 : Classification Binaire (Photo / Not-Photo)

## Partie 1.1 Classification Binaire sans Class Weights (CNN Personnalisé)

Cette section entraîne un CNN personnalisé pour classifier les images en deux catégories : **Photo** et **Not-Photo**, sans pondération des classes.

Le dataset est découpé en **60% train / 20% validation / 20% test**.

In [ ]:
import gc
import os
import pathlib
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import confusion_matrix, classification_report

### Initialisation de l'environnement GPU

On active la croissance dynamique de la VRAM pour éviter les erreurs *Out Of Memory*.

In [ ]:
def initialize_gpu_environment():
    """Configures TensorFlow to allocate VRAM dynamically to prevent OOM errors."""
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU Memory Growth enabled on {len(gpus)} device(s).")
        except RuntimeError as e:
            print(f"Memory growth configuration failed: {e}")
    else:
        print("No GPU detected. Falling back to CPU execution.")

initialize_gpu_environment()

### Variables Globales

**Adapter `DATASET_DIR`, `MODEL_SAVE_DIR` et `LOG_DIR` à votre environnement.**

Le split 60/20/20 est obtenu en chargeant d'abord 80% du dataset (train+val avec `validation_split=0.25` sur les 80%), puis en réservant les 20% restants pour le test.

In [ ]:
# Directory Routing
DATASET_DIR    = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/datasets/"
MODEL_SAVE_DIR = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/binary_model_run"
LOG_DIR        = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/logs/binary_model_run"

# Image Dimensions 
IMG_HEIGHT = 128
IMG_WIDTH  = 128
CHANNELS   = 3

# Training Hyperparameters
BATCH_SIZE = 16
EPOCHS     = 50
SEED       = 42

# Split ratios: 60% train / 20% val / 20% test
# image_dataset_from_directory splits the whole dataset.
# We first carve out 20% as a held-out test set (TEST_SPLIT).
# Among the remaining 80%, we use 25% for validation: 80%×25% = 20% overall.
TEST_SPLIT = 0.2
VAL_SPLIT  = 0.25

CLASS_NAMES = ["Not-Photo", "Photo"]
TIMESTAMP   = datetime.now().strftime("%d-%m-%Y-%H-%M-%S")

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

### Nettoyage du Dataset

Avant l'entraînement, on élimine les fichiers corrompus via un décodeur binaire strict TensorFlow exécuté en parallèle.

In [ ]:
def verify_and_clean_image(image_path, dry_run=True):
    """Evaluates image structural integrity using TensorFlow's native decoder."""
    try:
        img_raw = tf.io.read_file(str(image_path))
        decoded_img = tf.io.decode_jpeg(img_raw, channels=3)
        _ = decoded_img.shape
        return False
    except Exception as e:
        log_prefix = "[DRY RUN - WILL BE REMOVED]" if dry_run else "[DELETED]"
        print(f"{log_prefix} Corrupt image: {image_path.name} ({e})")
        if not dry_run:
            try:
                os.remove(image_path)
            except Exception as rm_err:
                print(f"Failed to remove {image_path.name}: {rm_err}")
        return True


def execute_dataset_sanitization(directory_path, dry_run=True):
    """Parses target directory asynchronously using a multi-threaded cleaning pipeline."""
    data_path = pathlib.Path(directory_path)
    if not data_path.exists():
        print(f"Error: Target directory '{directory_path}' does not exist!")
        return
    print(f"Analyzing directory: {data_path} (dry_run={dry_run})")
    image_extensions = [".jpg", ".jpeg", ".png"]
    all_image_paths = [
        p for p in data_path.rglob("*")
        if p.suffix.lower() in image_extensions
    ]
    print(f"Found {len(all_image_paths)} images to verify...")
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(
            lambda p: verify_and_clean_image(p, dry_run=dry_run),
            all_image_paths
        ))
    action = "detected" if dry_run else "removed"
    print(f"Sanitization complete: {sum(results)} corrupt files {action}.")
    del all_image_paths, results
    gc.collect()

execute_dataset_sanitization(DATASET_DIR, dry_run=False)

### Chargement et Préparation du Dataset (60 / 20 / 20)

- **Train** : 60% des données, utilisé pour l'entraînement.
- **Validation** : 20%, suivi de `val_loss` et `val_accuracy` pendant `model.fit`.
- **Test** : 20%, évaluation finale **après** entraînement jamais vu par le modèle.

In [ ]:
def load_binary_datasets():
    """
    Loads three independent dataset splits:
      - train  : 60% of the full dataset
      - val    : 20% of the full dataset (used during training)
      - test   : 20% of the full dataset (held-out, evaluated after training)
    """
    common_kwargs = dict(
        directory=DATASET_DIR,
        seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
    )

    trainval_raw = tf.keras.preprocessing.image_dataset_from_directory(
        **common_kwargs,
        validation_split=TEST_SPLIT,
        subset="training",        # 80%
    )
    test_raw = tf.keras.preprocessing.image_dataset_from_directory(
        **common_kwargs,
        validation_split=TEST_SPLIT,
        subset="validation",      # 20%
    )

    train_raw = tf.keras.preprocessing.image_dataset_from_directory(
        **common_kwargs,
        validation_split=VAL_SPLIT,
        subset="training",        # 75%
    )
    val_raw = tf.keras.preprocessing.image_dataset_from_directory(
        **common_kwargs,
        validation_split=VAL_SPLIT,
        subset="validation",      # 25% of 80% = 20%
    )

    return train_raw, val_raw, test_raw


bin_train_raw, bin_val_raw, bin_test_raw = load_binary_datasets()

raw_class_names = bin_train_raw.class_names
print("Original directory classes detected:", raw_class_names)
PHOTO_INDEX = raw_class_names.index("Photo")


def binarize_labels_tensor(images, labels):
    """Maps multi-class folder labels into binary 1.0 (Photo) / 0.0 (Not-Photo)."""
    binary_labels = tf.cast(tf.equal(labels, PHOTO_INDEX), tf.float32)
    return images, tf.expand_dims(binary_labels, axis=-1)


AUTOUNE = tf.data.AUTOTUNE

bin_train_set = bin_train_raw.map(binarize_labels_tensor).shuffle(200, seed=SEED).prefetch(AUTOUNE)
bin_val_set   = bin_val_raw.map(binarize_labels_tensor).prefetch(AUTOUNE)
bin_test_set  = bin_test_raw.map(binarize_labels_tensor).prefetch(AUTOUNE)

print(f"Binary labels applied: {CLASS_NAMES}")
print(f"Split Train: ~60% | Val: ~20% | Test: ~20%")

### Vérification Visuelle du Dataset

On affiche un échantillon d'images pour valider la binarisation des labels.

In [ ]:
def visualize_binary_samples(dataset, target_classes, grid_size=9):
    """Extracts a batch slice to verify image-to-binary-label mapping."""
    plt.figure(figsize=(8, 8))
    for images, labels in dataset.take(1):
        num_images = min(grid_size, len(images))
        for i in range(num_images):
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            class_idx = int(labels[i][0])
            plt.title(target_classes[class_idx])
            plt.axis("off")
    plt.tight_layout()
    plt.show()

visualize_binary_samples(bin_train_set, CLASS_NAMES)

### Data Augmentation

Les transformations aléatoires (flip, rotation, zoom, contraste) augmentent la diversité du dataset d'entraînement et réduisent le sur-apprentissage.

In [ ]:
bin_data_augmentation = Sequential([
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(factor=0.2),
], name="binary_augmentation")


def plot_augmentation_effect(dataset):
    """Shows one Photo image and 9 augmented variations."""
    target_image = None
    for images, labels in dataset:
        photo_indices = tf.where(labels == 1)
        if tf.size(photo_indices) > 0:
            target_image = images[photo_indices[0][0]]
            break
    if target_image is None:
        print("Warning: No 'Photo' (label=1) image found in this batch.")
        return
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 5, 1)
    plt.imshow(target_image.numpy().astype("uint8"))
    plt.title("Original", weight="bold", color="darkgreen")
    plt.axis("off")
    for i in range(9):
        aug = bin_data_augmentation(tf.expand_dims(target_image, 0), training=True)
        plt.subplot(2, 5, i + 2)
        plt.imshow(aug[0].numpy().astype("uint8"))
        plt.title(f"Variation {i+1}")
        plt.axis("off")
    plt.suptitle("Impact de la Data Augmentation", fontsize=14, weight="bold")
    plt.tight_layout()
    plt.show()

plot_augmentation_effect(bin_train_raw.map(binarize_labels_tensor))

### Architecture CNN Personnalisée

Le modèle est un CNN séquentiel avec 3 blocs Conv2D + BatchNorm + MaxPooling + Dropout, suivi d'une tête dense avec sortie binaire (sigmoïde via `BinaryCrossentropy(from_logits=True)`).

**Dimensionnement des Feature Maps :**

$$W_{out} = \lfloor \frac{W_{in}}{2} \rfloor$$

- Entrée : $128 \times 128$
- Après Pooling 1 : $64 \times 64$
- Après Pooling 2 : $32 \times 32$
- Après Pooling 3 : $16 \times 16$

In [ ]:
def build_binary_cnn_model(input_shape, augmentation_stack):
    """Assembles a sequential binary CNN with regularization (BatchNorm + Dropout)."""
    model = Sequential([
        augmentation_stack,
        layers.Rescaling(1.0 / 255, input_shape=input_shape),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1),   # logit output for BinaryCrossentropy(from_logits=True)
    ], name="binary_cnn")
    return model


bin_model_noweight = build_binary_cnn_model(
    (IMG_HEIGHT, IMG_WIDTH, CHANNELS), bin_data_augmentation
)
bin_model_noweight.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
bin_model_noweight.summary()

### Entraînement Binaire sans Class Weights

Callbacks utilisés :
- **EarlyStopping** : arrêt si `val_loss` ne s'améliore plus pendant 8 epochs.
- **ModelCheckpoint** : sauvegarde du meilleur modèle.
- **ReduceLROnPlateau** : réduction du learning rate si stagnation.
- **TensorBoard** : journalisation pour visualisation interactive.

In [ ]:
bin_log_dir_noweight = os.path.join(LOG_DIR, f"{TIMESTAMP}_noweight")
bin_checkpoint_noweight = os.path.join(MODEL_SAVE_DIR, f"best_binary_noweight-{TIMESTAMP}.keras")

bin_callbacks_noweight = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=bin_checkpoint_noweight, monitor="val_loss",
        save_best_only=True, mode="min", verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=1, min_lr=1e-6),
    tf.keras.callbacks.TensorBoard(
        log_dir=bin_log_dir_noweight, histogram_freq=1,
        write_graph=True, update_freq="epoch"
    ),
]

bin_history_noweight = bin_model_noweight.fit(
    bin_train_set,
    epochs=EPOCHS,
    validation_data=bin_val_set,
    callbacks=bin_callbacks_noweight,
)

print(f"\nTensorBoard logs: {bin_log_dir_noweight}")
print("Run: tensorboard --logdir", LOG_DIR)

### Résultats Courbes, Métriques et Matrice de Confusion

Évaluation finale sur le **test set** (20% du dataset, jamais utilisé pendant l'entraînement).

In [ ]:
def plot_training_curves_binary(history, model, test_dataset, class_names, save_dir, ts, log_dir):
    """Plots accuracy/loss curves, evaluates on test set, and builds confusion matrix."""
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))

    print(f"TensorBoard Log Dir     : {log_dir}")
    print(f"Training Loss (last)    : {loss[-1]:.4f}")
    print(f"Training Accuracy (last): {acc[-1]*100:.2f}%")
    print(f"Val Loss (last epoch)   : {val_loss[-1]:.4f}")
    print(f"Val Accuracy (last)     : {val_acc[-1]*100:.2f}%")

    # Evaluate on the held-out test set
    test_results = model.evaluate(test_dataset, verbose=0)
    print(f"\n→ TEST SET Loss     : {test_results[0]:.4f}")
    print(f"→ TEST SET Accuracy : {test_results[1]*100:.2f}%")

    os.makedirs(save_dir, exist_ok=True)

    # Training curves
    plt.figure(figsize=(16, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc,     label="Train Accuracy")
    plt.plot(epochs_range, val_acc, label="Val Accuracy")
    plt.legend(loc="lower right")
    plt.title("Accuracy Train vs Validation")
    plt.xlabel("Epochs"); plt.ylabel("Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss,     label="Train Loss")
    plt.plot(epochs_range, val_loss, label="Val Loss")
    plt.legend(loc="upper right")
    plt.title("Loss Train vs Validation")
    plt.xlabel("Epochs"); plt.ylabel("Loss")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"curves_{ts}.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Confusion matrix on test set
    print("\nBuilding Confusion Matrix on TEST SET...")
    true_labels, predicted_labels = [], []
    for imgs, lbls in test_dataset:
        logits = model.predict(imgs, verbose=0)
        preds  = (tf.nn.sigmoid(logits).numpy() >= 0.5).astype(int).flatten()
        true_labels.extend(lbls.numpy().flatten().astype(int))
        predicted_labels.extend(preds)

    true_labels      = np.array(true_labels)
    predicted_labels = np.array(predicted_labels)
    cm = confusion_matrix(true_labels, predicted_labels)

    cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix Test Set (Binary)")
    plt.ylabel("True Label"); plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"cm_{ts}.png"), dpi=150, bbox_inches="tight")
    plt.show()

    print("\nClassification Report (Test Set):")
    print(classification_report(true_labels, predicted_labels, target_names=class_names))

    del true_labels, predicted_labels, cm
    gc.collect()


plot_training_curves_binary(bin_history_noweight, bin_model_noweight, bin_test_set,CLASS_NAMES, MODEL_SAVE_DIR, f"{TIMESTAMP}_noweight", bin_log_dir_noweight)

### Analyse Qualitative Succès et Erreurs du Modèle

On isole les **vrais positifs / vrais négatifs** (succès) et les **faux positifs / faux négatifs** (erreurs) avec leur score de confiance.

In [ ]:
def execute_binary_error_analysis(model, dataset, target_classes, save_dir, ts):
    """Isolates TP, TN, FP, FN samples and plots them with confidence scores."""
    all_images, all_labels = [], []
    for imgs, lbls in dataset.unbatch().take(5 * BATCH_SIZE).as_numpy_iterator():
        all_images.append(imgs)
        all_labels.append(lbls)

    all_images = np.array(all_images).astype("uint8")
    all_labels = np.array(all_labels).astype(int).flatten()

    logits      = model.predict(all_images, batch_size=BATCH_SIZE, verbose=1)
    all_probs   = tf.nn.sigmoid(logits).numpy().flatten()
    pred_classes = (all_probs >= 0.5).astype(int)
    confidences  = np.where(pred_classes == 1, all_probs, 1 - all_probs)

    tn_idx = np.where((all_labels == 0) & (pred_classes == 0))[0]
    tp_idx = np.where((all_labels == 1) & (pred_classes == 1))[0]
    fp_idx = np.where((all_labels == 0) & (pred_classes == 1))[0]
    fn_idx = np.where((all_labels == 1) & (pred_classes == 0))[0]

    sorted_tn = tn_idx[np.argsort(confidences[tn_idx])[::-1]]
    sorted_tp = tp_idx[np.argsort(confidences[tp_idx])[::-1]]
    sorted_fp = fp_idx[np.argsort(confidences[fp_idx])[::-1]]
    sorted_fn = fn_idx[np.argsort(confidences[fn_idx])[::-1]]

    # Success plot 
    plt.figure(figsize=(16, 8))
    plt.suptitle("Top Model Successes (Highest Confidence)", fontsize=16, weight="bold")
    for i in range(min(3, len(sorted_tn))):
        idx = sorted_tn[i]
        plt.subplot(2, 3, i + 1)
        plt.imshow(all_images[idx])
        plt.title(f"True: {target_classes[all_labels[idx]]}\nPred: {target_classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="green", fontsize=9)
        plt.axis("off")
    for i in range(min(3, len(sorted_tp))):
        idx = sorted_tp[i]
        plt.subplot(2, 3, i + 4)
        plt.imshow(all_images[idx])
        plt.title(f"True: {target_classes[all_labels[idx]]}\nPred: {target_classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="green", fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"successes_{ts}.png"), dpi=150)
    plt.show()

    # Failure plot 
    plt.figure(figsize=(16, 8))
    plt.suptitle("Model Classification Failures (FP + FN)", fontsize=16, weight="bold")
    for i in range(min(3, len(sorted_fp))):
        idx = sorted_fp[i]
        plt.subplot(2, 3, i + 1)
        plt.imshow(all_images[idx])
        plt.title(f"True: {target_classes[all_labels[idx]]}\nPred: {target_classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="red", fontsize=9)
        plt.axis("off")
    for i in range(min(3, len(sorted_fn))):
        idx = sorted_fn[i]
        plt.subplot(2, 3, i + 4)
        plt.imshow(all_images[idx])
        plt.title(f"True: {target_classes[all_labels[idx]]}\nPred: {target_classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="red", fontsize=9)
        plt.axis("off")
    if len(sorted_fp) == 0: print("No False Positives detected.")
    if len(sorted_fn) == 0: print("No False Negatives detected.")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"failures_{ts}.png"), dpi=150)
    plt.show()

    del all_images, all_labels, all_probs, pred_classes, confidences
    gc.collect()


execute_binary_error_analysis(bin_model_noweight, bin_test_set, CLASS_NAMES,MODEL_SAVE_DIR, f"{TIMESTAMP}_noweight")

 
## Partie 1.2 Classification Binaire avec Class Weights (CNN Personnalisé)

Même architecture CNN, mais avec des **class weights** calculés selon la fréquence inverse des classes pour corriger le déséquilibre du dataset.

In [ ]:
import gc
import os
import pathlib
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Directory Routing
DATASET_DIR    = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/datasets/"
MODEL_SAVE_DIR = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/binary_model_run"
LOG_DIR        = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/logs/binary_model_run"

IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 16  
EPOCHS = 50   
SEED = 42
TEST_SPLIT = 0.2
VAL_SPLIT = 0.25
CLASS_NAMES = ["Not-Photo", "Photo"]
TIMESTAMP   = datetime.now().strftime("%d-%m-%Y-%H-%M-%S")
AUTOUNE     = tf.data.AUTOTUNE

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
def load_binary_datasets_p12():
    """Loads three independent splits: 60% train / 20% val / 20% test (binary)."""
    common_kwargs = dict(
        directory=DATASET_DIR, seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    )
    train_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="training")
    val_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="validation")
    test_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=TEST_SPLIT, subset="validation")
    return train_raw, val_raw, test_raw

bin_train_raw_w, bin_val_raw_w, bin_test_raw_w = load_binary_datasets_p12()
raw_class_names_w = bin_train_raw_w.class_names
PHOTO_INDEX_W = raw_class_names_w.index("Photo")

def binarize_labels_p12(images, labels):
    """Maps multi-class labels to binary 1.0 (Photo) / 0.0 (Not-Photo)."""
    binary_labels = tf.cast(tf.equal(labels, PHOTO_INDEX_W), tf.float32)
    return images, tf.expand_dims(binary_labels, axis=-1)

bin_train_set_w = bin_train_raw_w.map(binarize_labels_p12).shuffle(200, seed=SEED).prefetch(AUTOUNE)
bin_val_set_w   = bin_val_raw_w.map(binarize_labels_p12).prefetch(AUTOUNE)
bin_test_set_w  = bin_test_raw_w.map(binarize_labels_p12).prefetch(AUTOUNE)
print(f"Split Train: ~60% | Val: ~20% | Test: ~20%  |  Classes: {CLASS_NAMES}")

### Calcul des Class Weights

La pondération standard par fréquence inverse :
$$w_c = \frac{N_{total}}{N_{classes} \times N_c}$$

Cela donne plus d'importance aux classes sous-représentées.

In [ ]:
y_train_w     = np.concatenate([y.numpy().flatten() for _, y in bin_train_set_w], axis=0)
total_w       = len(y_train_w)
count_0       = np.sum(y_train_w == 0)
count_1       = np.sum(y_train_w == 1)
weight_for_0  = total_w / (2.0 * count_0) if count_0 > 0 else 1.0
weight_for_1  = total_w / (2.0 * count_1) if count_1 > 0 else 1.0
binary_class_weights = {0: weight_for_0, 1: weight_for_1}

print(f"{CLASS_NAMES[0]}: {count_0} samples | {CLASS_NAMES[1]}: {count_1} samples")
print(f"Class weights: {binary_class_weights}")

In [ ]:
bin_data_augmentation_w = Sequential([
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(factor=0.2),
], name="binary_augmentation_weighted")


def build_binary_cnn_model_p12(input_shape, augmentation_stack):
    """Assembles the same binary CNN architecture for the weighted variant."""
    model = Sequential([
        augmentation_stack,
        layers.Rescaling(1.0 / 255, input_shape=input_shape),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.2),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.2),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(1),
    ], name="binary_cnn_weighted")
    return model


bin_model_weighted = build_binary_cnn_model_p12(
    (IMG_HEIGHT, IMG_WIDTH, CHANNELS), bin_data_augmentation_w
)
bin_model_weighted.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
bin_model_weighted.summary()

In [ ]:
bin_log_dir_w    = os.path.join(LOG_DIR, f"{TIMESTAMP}_weighted")
bin_checkpoint_w = os.path.join(MODEL_SAVE_DIR, f"best_binary_weighted-{TIMESTAMP}.keras")

bin_callbacks_w = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=bin_checkpoint_w, monitor="val_loss",
        save_best_only=True, mode="min", verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, verbose=1, min_lr=1e-6
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir=bin_log_dir_w, histogram_freq=1,
        write_graph=True, update_freq="epoch"
    ),
]

bin_history_w = bin_model_weighted.fit(
    bin_train_set_w,
    epochs=EPOCHS,
    validation_data=bin_val_set_w,
    callbacks=bin_callbacks_w,
    class_weight=binary_class_weights,
)
print(f"\nTensorBoard logs: {bin_log_dir_w}")

### Résultats Avec Class Weights

In [ ]:
def plot_training_curves_binary_w(history, model, test_dataset, class_names, save_dir, ts, log_dir):
    """Same as plot_training_curves_binary but for the weighted model."""
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))

    print(f"TensorBoard Log Dir     : {log_dir}")
    print(f"Training Loss (last)    : {loss[-1]:.4f}")
    print(f"Val Loss (last)         : {val_loss[-1]:.4f}")
    print(f"Val Accuracy (last)     : {val_acc[-1]*100:.2f}%")

    test_results = model.evaluate(test_dataset, verbose=0)
    print(f"\n→ TEST SET Loss     : {test_results[0]:.4f}")
    print(f"→ TEST SET Accuracy : {test_results[1]*100:.2f}%")

    os.makedirs(save_dir, exist_ok=True)
    plt.figure(figsize=(16, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Train"); plt.plot(epochs_range, val_acc, label="Val")
    plt.legend(); plt.title("Accuracy Weighted Binary"); plt.xlabel("Epochs")
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Train"); plt.plot(epochs_range, val_loss, label="Val")
    plt.legend(); plt.title("Loss Weighted Binary"); plt.xlabel("Epochs")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"curves_{ts}.png"), dpi=150, bbox_inches="tight")
    plt.show()

    true_labels, predicted_labels = [], []
    for imgs, lbls in test_dataset:
        logits = model.predict(imgs, verbose=0)
        preds  = (tf.nn.sigmoid(logits).numpy() >= 0.5).astype(int).flatten()
        true_labels.extend(lbls.numpy().flatten().astype(int))
        predicted_labels.extend(preds)
    true_labels      = np.array(true_labels)
    predicted_labels = np.array(predicted_labels)
    cm = confusion_matrix(true_labels, predicted_labels)

    cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix Test Set (Binary Weighted)")
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"cm_{ts}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("\nClassification Report:")
    print(classification_report(true_labels, predicted_labels, target_names=class_names))
    del true_labels, predicted_labels, cm
    gc.collect()


plot_training_curves_binary_w(bin_history_w, bin_model_weighted, bin_test_set_w,CLASS_NAMES, MODEL_SAVE_DIR, f"{TIMESTAMP}_weighted", bin_log_dir_w)

In [ ]:
def execute_binary_error_analysis_p12(model, dataset, target_classes, save_dir, ts):
    """Isolates TP, TN, FP, FN for the weighted binary model."""
    all_images, all_labels = [], []
    for imgs, lbls in dataset.unbatch().take(5 * BATCH_SIZE).as_numpy_iterator():
        all_images.append(imgs); all_labels.append(lbls)
    all_images   = np.array(all_images).astype("uint8")
    all_labels   = np.array(all_labels).astype(int).flatten()
    logits       = model.predict(all_images, batch_size=BATCH_SIZE, verbose=1)
    all_probs    = tf.nn.sigmoid(logits).numpy().flatten()
    pred_classes = (all_probs >= 0.5).astype(int)
    confidences  = np.where(pred_classes == 1, all_probs, 1 - all_probs)
    tn_idx = np.where((all_labels == 0) & (pred_classes == 0))[0]
    tp_idx = np.where((all_labels == 1) & (pred_classes == 1))[0]
    fp_idx = np.where((all_labels == 0) & (pred_classes == 1))[0]
    fn_idx = np.where((all_labels == 1) & (pred_classes == 0))[0]
    for group, label, color, slot in [
        (tn_idx, "True Negatives", "green", range(1, 4)),
        (tp_idx, "True Positives", "green", range(4, 7)),
    ]:
        sorted_g = group[np.argsort(confidences[group])[::-1]]
        plt.figure(figsize=(16, 8)); plt.suptitle(f"Successes {label}", fontsize=14, weight="bold")
        for i, s in zip(range(min(3, len(sorted_g))), slot):
            idx = sorted_g[i]
            plt.subplot(1, 3, i + 1); plt.imshow(all_images[idx])
            plt.title(f"True:{target_classes[all_labels[idx]]}\nPred:{target_classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color=color, fontsize=9)
            plt.axis("off")
        plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"{label.replace(' ','_')}_{ts}.png"), dpi=150); plt.show()
    if len(fp_idx) == 0: print("No False Positives.")
    if len(fn_idx) == 0: print("No False Negatives.")
    del all_images, all_labels, all_probs, pred_classes, confidences
    gc.collect()

execute_binary_error_analysis_p12(bin_model_weighted, bin_test_set_w, CLASS_NAMES,MODEL_SAVE_DIR, f"{TIMESTAMP}_weighted")

 
# Partie 2 : Classification Multi-Classes à 5 Classes (CNN Personnalisé)
 

## Partie 2.1 Multi-Classes sans Class Weights

Classification des images en **5 classes** avec un CNN personnalisé, sans pondération. Split **60/20/20**.

In [ ]:
import gc
import os
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Directory Routing
DATASET_DIR    = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/datasets/"
MODEL_SAVE_DIR = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/multi_class_model_run"
LOG_DIR        = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/logs/multi_class_model_run"

IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 16; 
EPOCHS = 50;   
SEED = 42
TEST_SPLIT = 0.2
VAL_SPLIT = 0.25
AUTOUNE    = tf.data.AUTOTUNE
TIMESTAMP  = datetime.now().strftime("%d-%m-%Y-%H-%M-%S")

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
def load_mc_datasets():
    """Loads three independent splits (60/20/20) for multi-class classification."""
    common_kwargs = dict(
        directory=DATASET_DIR, seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    )
    train_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="training")
    val_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="validation")
    test_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=TEST_SPLIT, subset="validation")
    return train_raw, val_raw, test_raw


mc_train_raw, mc_val_raw, mc_test_raw = load_mc_datasets()

MC_CLASS_NAMES = mc_train_raw.class_names
NUM_CLASSES    = len(MC_CLASS_NAMES)

mc_train_set = mc_train_raw.shuffle(200, seed=SEED).prefetch(AUTOUNE)
mc_val_set   = mc_val_raw.prefetch(AUTOUNE)
mc_test_set  = mc_test_raw.prefetch(AUTOUNE)

print(f"Multi-Class Pipeline Ready: {NUM_CLASSES} classes {MC_CLASS_NAMES}")
print(f"Split Train: ~60% | Val: ~20% | Test: ~20%")

### Vérification Visuelle et Data Augmentation

In [ ]:
def visualize_mc_samples(dataset, target_classes, grid_size=9):
    """Extracts a batch slice to verify multi-class image-to-label mapping."""
    plt.figure(figsize=(8, 8))
    for images, labels in dataset.take(1):
        n = min(grid_size, len(images))
        for i in range(n):
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(target_classes[int(labels[i])])
            plt.axis("off")
    plt.tight_layout(); plt.show()

visualize_mc_samples(mc_train_set, MC_CLASS_NAMES)

mc_data_augmentation = Sequential([
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(factor=0.2),
], name="mc_augmentation")

### Architecture CNN Multi-Classes

Même structure que le modèle binaire, avec une sortie `Dense(num_classes)` et `SparseCategoricalCrossentropy(from_logits=True)`.

In [ ]:
def build_mc_cnn_model(input_shape, num_classes, augmentation_stack):
    """Builds a sequential CNN for multi-class categorical classification."""
    model = Sequential([
        augmentation_stack,
        layers.Rescaling(1.0 / 255, input_shape=input_shape),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.2),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.2),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(num_classes),  # logits for SparseCategoricalCrossentropy
    ], name="mc_cnn")
    return model


mc_model_noweight = build_mc_cnn_model(
    (IMG_HEIGHT, IMG_WIDTH, CHANNELS), NUM_CLASSES, mc_data_augmentation
)
mc_model_noweight.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
mc_model_noweight.summary()

In [ ]:
mc_log_dir_nw    = os.path.join(LOG_DIR, f"{TIMESTAMP}_noweight")
mc_checkpoint_nw = os.path.join(MODEL_SAVE_DIR, f"best_mc_noweight-{TIMESTAMP}.keras")

mc_callbacks_nw = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=mc_checkpoint_nw, monitor="val_loss",
        save_best_only=True, mode="min", verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=1, min_lr=1e-6),
    tf.keras.callbacks.TensorBoard(
        log_dir=mc_log_dir_nw, histogram_freq=1,
        write_graph=True, update_freq="epoch"
    ),
]

mc_history_nw = mc_model_noweight.fit(
    mc_train_set,
    epochs=EPOCHS,
    validation_data=mc_val_set,
    callbacks=mc_callbacks_nw,
)
print(f"\nTensorBoard logs: {mc_log_dir_nw}")

### Résultats Courbes, Test Set et Matrice de Confusion

In [ ]:
def plot_training_curves_mc(history, model, test_dataset, class_names, save_dir, ts, log_dir):
    """Plots accuracy/loss curves, evaluates on test set, and builds multi-class confusion matrix."""
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))

    print(f"TensorBoard Log Dir : {log_dir}")
    print(f"Val Loss (last)     : {val_loss[-1]:.4f}")
    print(f"Val Accuracy (last) : {val_acc[-1]*100:.2f}%")

    test_results = model.evaluate(test_dataset, verbose=0)
    print(f"\n→ TEST SET Loss     : {test_results[0]:.4f}")
    print(f"→ TEST SET Accuracy : {test_results[1]*100:.2f}%")

    os.makedirs(save_dir, exist_ok=True)
    plt.figure(figsize=(16, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Train"); plt.plot(epochs_range, val_acc, label="Val")
    plt.legend(); plt.title("Accuracy Multi-Class CNN"); plt.xlabel("Epochs")
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Train"); plt.plot(epochs_range, val_loss, label="Val")
    plt.legend(); plt.title("Loss Multi-Class CNN"); plt.xlabel("Epochs")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"curves_{ts}.png"), dpi=150, bbox_inches="tight")
    plt.show()

    print("\nBuilding Confusion Matrix on TEST SET...")
    true_labels, predicted_labels = [], []
    for imgs, lbls in test_dataset:
        logits = model.predict(imgs, verbose=0)
        preds  = np.argmax(tf.nn.softmax(logits).numpy(), axis=1)
        true_labels.extend(lbls.numpy().flatten().astype(int))
        predicted_labels.extend(preds)
    true_labels      = np.array(true_labels)
    predicted_labels = np.array(predicted_labels)
    cm = confusion_matrix(true_labels, predicted_labels)

    cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix Test Set (Multi-Class)")
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"cm_{ts}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("\nClassification Report (Test Set):")
    print(classification_report(true_labels, predicted_labels, target_names=class_names))
    del true_labels, predicted_labels, cm
    gc.collect()

plot_training_curves_mc(mc_history_nw, mc_model_noweight, mc_test_set,MC_CLASS_NAMES, MODEL_SAVE_DIR, f"{TIMESTAMP}_noweight", mc_log_dir_nw)

In [ ]:
def execute_mc_error_analysis(model, dataset, classes, save_dir, ts):
    """Isolates categorical successes and failures for the multi-class model."""
    all_images, all_labels, all_probs = [], [], []
    for imgs, lbls in dataset.take(5):
        logits = model.predict(imgs, verbose=0)
        probs  = tf.nn.softmax(logits).numpy()
        all_images.append(imgs.numpy())
        all_labels.append(lbls.numpy().flatten())
        all_probs.append(probs)
    all_images   = np.concatenate(all_images).astype("uint8")
    all_labels   = np.concatenate(all_labels).astype(int)
    all_probs    = np.concatenate(all_probs)
    pred_classes = np.argmax(all_probs, axis=1)
    confidences  = np.max(all_probs, axis=1)
    success_idx  = np.where(pred_classes == all_labels)[0]
    failure_idx  = np.where(pred_classes != all_labels)[0]
    sorted_s = success_idx[np.argsort(confidences[success_idx])[::-1]]
    sorted_f = failure_idx[np.argsort(confidences[failure_idx])[::-1]]

    plt.figure(figsize=(16, 8))
    plt.suptitle("Multi-Class Successes (Highest Confidence)", fontsize=16, weight="bold")
    for i in range(min(6, len(sorted_s))):
        idx = sorted_s[i]
        plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
        plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="green", fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"mc_success_{ts}.png"), dpi=150); plt.show()

    if len(sorted_f) > 0:
        plt.figure(figsize=(16, 8))
        plt.suptitle("Multi-Class Failures (Inter-Class Confusion)", fontsize=16, weight="bold")
        for i in range(min(6, len(sorted_f))):
            idx = sorted_f[i]
            plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
            plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="red", fontsize=9)
            plt.axis("off")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"mc_failures_{ts}.png"), dpi=150); plt.show()
    else:
        print("No multi-class errors detected on this sample.")

    del all_images, all_labels, all_probs, pred_classes, confidences
    gc.collect()


execute_mc_error_analysis(mc_model_noweight, mc_test_set, MC_CLASS_NAMES,MODEL_SAVE_DIR, f"{TIMESTAMP}_noweight")

 
## Partie 2.2 Multi-Classes avec Class Weights

In [ ]:
import gc
import os
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
DATASET_DIR    = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/datasets/"
MODEL_SAVE_DIR = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/multi_class_model_run"
LOG_DIR        = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/logs/multi_class_model_run"
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 50
SEED = 42
TEST_SPLIT = 0.2
VAL_SPLIT = 0.25
AUTOUNE    = tf.data.AUTOTUNE
TIMESTAMP  = datetime.now().strftime("%d-%m-%Y-%H-%M-%S")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
def load_mc_datasets_w():
    """Loads three independent splits (60/20/20) for weighted multi-class training."""
    common_kwargs = dict(
        directory=DATASET_DIR, seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    )
    train_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="training")
    val_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="validation")
    test_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=TEST_SPLIT, subset="validation")
    return train_raw, val_raw, test_raw

mc_train_raw_w, mc_val_raw_w, mc_test_raw_w = load_mc_datasets_w()
MC_CLASS_NAMES_W = mc_train_raw_w.class_names
NUM_CLASSES_W    = len(MC_CLASS_NAMES_W)

mc_train_set_w = mc_train_raw_w.shuffle(200, seed=SEED).prefetch(AUTOUNE)
mc_val_set_w   = mc_val_raw_w.prefetch(AUTOUNE)
mc_test_set_w  = mc_test_raw_w.prefetch(AUTOUNE)
print(f"Classes: {MC_CLASS_NAMES_W} | Split Train: ~60% | Val: ~20% | Test: ~20%")

In [ ]:
y_train_mc_w   = np.concatenate([y.numpy().flatten() for _, y in mc_train_set_w])
total_mc       = len(y_train_mc_w)
mc_class_weights = {}
for i in range(NUM_CLASSES_W):
    count = np.sum(y_train_mc_w == i)
    mc_class_weights[i] = total_mc / (NUM_CLASSES_W * count) if count > 0 else 1.0
    print(f"  Class {i} ({MC_CLASS_NAMES_W[i]}): {count} samples | weight = {mc_class_weights[i]:.4f}")
print(f"\nClass weights: {mc_class_weights}")

In [ ]:
mc_data_augmentation_w = Sequential([
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(factor=0.2),
], name="mc_augmentation_weighted")


def build_mc_cnn_model_w(input_shape, num_classes, augmentation_stack):
    """Builds the same multi-class CNN for the weighted training variant."""
    model = Sequential([
        augmentation_stack,
        layers.Rescaling(1.0 / 255, input_shape=input_shape),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.2),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.2),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(num_classes),
    ], name="mc_cnn_weighted")
    return model


mc_model_weighted = build_mc_cnn_model_w(
    (IMG_HEIGHT, IMG_WIDTH, CHANNELS), NUM_CLASSES_W, mc_data_augmentation_w
)
mc_model_weighted.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
mc_model_weighted.summary()

In [ ]:
mc_log_dir_w    = os.path.join(LOG_DIR, f"{TIMESTAMP}_weighted")
mc_checkpoint_w = os.path.join(MODEL_SAVE_DIR, f"best_mc_weighted-{TIMESTAMP}.keras")

mc_callbacks_w = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=mc_checkpoint_w, monitor="val_loss",
        save_best_only=True, mode="min", verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=1, min_lr=1e-6),
    tf.keras.callbacks.TensorBoard(
        log_dir=mc_log_dir_w, histogram_freq=1,
        write_graph=True, update_freq="epoch"
    ),
]

mc_history_w = mc_model_weighted.fit(
    mc_train_set_w,
    epochs=EPOCHS,
    validation_data=mc_val_set_w,
    callbacks=mc_callbacks_w,
    class_weight=mc_class_weights,
)
print(f"\nTensorBoard logs: {mc_log_dir_w}")

In [ ]:
def plot_training_curves_mc_w(history, model, test_dataset, class_names, save_dir, ts, log_dir):
    """Plots curves, evaluates on test set, and builds confusion matrix for weighted multi-class model."""
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))
    print(f"Val Accuracy: {val_acc[-1]*100:.2f}% | Val Loss: {val_loss[-1]:.4f}")
    test_r = model.evaluate(test_dataset, verbose=0)
    print(f"→ TEST SET  Accuracy: {test_r[1]*100:.2f}% | Loss: {test_r[0]:.4f}")
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(16, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Train"); plt.plot(epochs_range, val_acc, label="Val")
    plt.legend(); plt.title("Accuracy Weighted Multi-Class")
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Train"); plt.plot(epochs_range, val_loss, label="Val")
    plt.legend(); plt.title("Loss Weighted Multi-Class")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"curves_{ts}.png"), dpi=150); plt.show()
    true_labels, predicted_labels = [], []

    for imgs, lbls in test_dataset:
        logits = model.predict(imgs, verbose=0)
        preds  = np.argmax(tf.nn.softmax(logits).numpy(), axis=1)
        true_labels.extend(lbls.numpy().flatten().astype(int))
        predicted_labels.extend(preds)

    true_labels = np.array(true_labels); predicted_labels = np.array(predicted_labels)
    cm = confusion_matrix(true_labels, predicted_labels)

    cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix Test Set (Weighted Multi-Class)")
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"cm_{ts}.png"), dpi=150); plt.show()
    print(classification_report(true_labels, predicted_labels, target_names=class_names))
    del true_labels, predicted_labels, cm; gc.collect()

plot_training_curves_mc_w(mc_history_w, mc_model_weighted, mc_test_set_w,MC_CLASS_NAMES_W, MODEL_SAVE_DIR, f"{TIMESTAMP}_weighted", mc_log_dir_w)

def execute_mc_error_analysis_w(model, dataset, classes, save_dir, ts):
    """Isolates successes and failures for the weighted multi-class model."""
    all_images, all_labels, all_probs = [], [], []
    for imgs, lbls in dataset.take(5):
        logits = model.predict(imgs, verbose=0)
        probs  = tf.nn.softmax(logits).numpy()
        all_images.append(imgs.numpy()); all_labels.append(lbls.numpy().flatten()); all_probs.append(probs)
    all_images   = np.concatenate(all_images).astype("uint8")
    all_labels   = np.concatenate(all_labels).astype(int)
    all_probs    = np.concatenate(all_probs)
    pred_classes = np.argmax(all_probs, axis=1)
    confidences  = np.max(all_probs, axis=1)
    success_idx  = np.where(pred_classes == all_labels)[0]
    failure_idx  = np.where(pred_classes != all_labels)[0]
    sorted_s = success_idx[np.argsort(confidences[success_idx])[::-1]]
    sorted_f = failure_idx[np.argsort(confidences[failure_idx])[::-1]]
    plt.figure(figsize=(16, 8))
    plt.suptitle("Weighted MC Top Successes", fontsize=16, weight="bold")
    for i in range(min(6, len(sorted_s))):
        idx = sorted_s[i]; plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
        plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="green", fontsize=9); plt.axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"mc_w_success_{ts}.png"), dpi=150); plt.show()
    if len(sorted_f) > 0:
        plt.figure(figsize=(16, 8))
        plt.suptitle("Weighted MC Failures", fontsize=16, weight="bold")
        for i in range(min(6, len(sorted_f))):
            idx = sorted_f[i]; plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
            plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="red", fontsize=9); plt.axis("off")
        plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"mc_w_failures_{ts}.png"), dpi=150); plt.show()
    del all_images, all_labels, all_probs, pred_classes, confidences; gc.collect()

execute_mc_error_analysis_w(mc_model_weighted, mc_test_set_w, MC_CLASS_NAMES_W,MODEL_SAVE_DIR, f"{TIMESTAMP}_weighted")

# Partie 3 : Transfer Learning VGG19 (Multi-Classes)


## Partie 3.1 VGG19 sans Class Weights

On charge un backbone **VGG19 pré-entraîné sur ImageNet** (couches gelées) et on ajoute une tête de classification personnalisée. Le taux d'apprentissage est volontairement bas ($\alpha = 0.0001$) pour ne pas détruire les poids pré-entraînés. Split **60/20/20**.

In [ ]:
import gc
import os
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Directory Routing
DATASET_DIR    = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/datasets/"
MODEL_SAVE_DIR = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/transfer_learning_run"
LOG_DIR        = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/logs/transfer_learning_run"

IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 50
SEED = 42
TEST_SPLIT = 0.2
VAL_SPLIT = 0.25
LR_TL      = 0.0001   # Low learning rate to preserve pretrained weights
AUTOUNE    = tf.data.AUTOTUNE
TIMESTAMP  = datetime.now().strftime("%d-%m-%Y-%H-%M-%S")

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
def load_tl_datasets():
    """Loads three independent splits (60/20/20) for transfer learning."""
    common_kwargs = dict(
        directory=DATASET_DIR, seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    )
    train_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="training")
    val_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="validation")
    test_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=TEST_SPLIT, subset="validation")
    return train_raw, val_raw, test_raw


tl_train_raw, tl_val_raw, tl_test_raw = load_tl_datasets()

TL_CLASS_NAMES = tl_train_raw.class_names
NUM_CLASSES_TL = len(TL_CLASS_NAMES)

tl_train_set = tl_train_raw.shuffle(200, seed=SEED).prefetch(AUTOUNE)
tl_val_set   = tl_val_raw.prefetch(AUTOUNE)
tl_test_set  = tl_test_raw.prefetch(AUTOUNE)

print(f"TL Pipeline Ready: {NUM_CLASSES_TL} classes {TL_CLASS_NAMES}")
print(f"Split Train: ~60% | Val: ~20% | Test: ~20%")

### Architecture VGG19 + Tête de Classification

Le backbone VGG19 est gelé (`trainable=False`) seule la tête dense est entraînée. Le preprocessing natif VGG19 remplace la couche `Rescaling`.

In [ ]:
tl_augmentation = Sequential([
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(factor=0.2),
], name="tl_augmentation")


def build_vgg19_model(input_shape, num_classes, augmentation_stack):
    """Instantiates VGG19 backbone (frozen) with a custom classification head."""
    base_vgg = tf.keras.applications.VGG19(
        include_top=False, weights="imagenet",
        input_shape=input_shape, pooling="avg"
    )
    base_vgg.trainable = False  # Freeze pretrained feature extraction layers

    model = Sequential([
        augmentation_stack,
        layers.Lambda(
            tf.keras.applications.vgg19.preprocess_input,
            name="vgg19_preprocessing"
        ),
        base_vgg,
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(num_classes),  # logits for SparseCategoricalCrossentropy
    ], name="vgg19_transfer_classifier")
    return model


vgg_model_nw = build_vgg19_model(
    (IMG_HEIGHT, IMG_WIDTH, CHANNELS), NUM_CLASSES_TL, tl_augmentation
)
vgg_model_nw.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_TL),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
vgg_model_nw.summary()

In [ ]:
tl_log_dir_nw    = os.path.join(LOG_DIR, f"{TIMESTAMP}_noweight")
tl_checkpoint_nw = os.path.join(MODEL_SAVE_DIR, f"best_vgg19_noweight-{TIMESTAMP}.keras")

tl_callbacks_nw = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=tl_checkpoint_nw, monitor="val_loss",
        save_best_only=True, mode="min", verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, verbose=1, min_lr=1e-7
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir=tl_log_dir_nw, histogram_freq=1,
        write_graph=True, update_freq="epoch"
    ),
]

print("Starting VGG19 Transfer Learning training...")
vgg_history_nw = vgg_model_nw.fit(
    tl_train_set,
    epochs=EPOCHS,
    validation_data=tl_val_set,
    callbacks=tl_callbacks_nw,
)
print(f"\nTensorBoard logs {tl_log_dir_nw}")

### Résultats VGG19 sans Class Weights

In [ ]:
def plot_tl_results(history, model, test_dataset, class_names, save_dir, ts, log_dir):
    """Plots VGG19 training curves, evaluates on test set, and builds confusion matrix."""
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))

    print(f"TensorBoard Log Dir : {log_dir}")
    print(f"Val Loss (last)     : {val_loss[-1]:.4f}")
    print(f"Val Accuracy (last) : {val_acc[-1]*100:.2f}%")

    test_r = model.evaluate(test_dataset, verbose=0)
    print(f"\n→ TEST SET Loss     : {test_r[0]:.4f}")
    print(f"→ TEST SET Accuracy : {test_r[1]*100:.2f}%")

    os.makedirs(save_dir, exist_ok=True)
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Train", color="royalblue")
    plt.plot(epochs_range, val_acc, label="Val", color="darkorange")
    plt.title("VGG19 Accuracy"); plt.xlabel("Epochs"); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Train", color="royalblue")
    plt.plot(epochs_range, val_loss, label="Val", color="darkorange")
    plt.title("VGG19 Loss"); plt.xlabel("Epochs"); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"vgg19_curves_{ts}.png"), dpi=150); plt.show()

    print("\nBuilding Confusion Matrix on TEST SET...")
    true_labels, predicted_labels = [], []
    for imgs, lbls in test_dataset:
        logits = model.predict(imgs, verbose=0)
        preds  = np.argmax(tf.nn.softmax(logits).numpy(), axis=1)
        true_labels.extend(lbls.numpy().flatten())
        predicted_labels.extend(preds)
    true_labels = np.array(true_labels); predicted_labels = np.array(predicted_labels)
    cm = confusion_matrix(true_labels, predicted_labels)

    cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix Test Set (VGG19)")
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"vgg19_cm_{ts}.png"), dpi=150); plt.show()
    print(classification_report(true_labels, predicted_labels, target_names=class_names))
    del true_labels, predicted_labels, cm; gc.collect()


plot_tl_results(
    vgg_history_nw, vgg_model_nw, tl_test_set,
    TL_CLASS_NAMES, MODEL_SAVE_DIR, f"{TIMESTAMP}_noweight", tl_log_dir_nw
)

In [ ]:
def execute_tl_error_analysis(model, dataset, classes, save_dir, ts):
    """Isolates VGG19 successes and failures with confidence scores."""
    all_images, all_labels, all_probs = [], [], []
    for imgs, lbls in dataset.take(5):
        logits = model.predict(imgs, verbose=0)
        probs  = tf.nn.softmax(logits).numpy()
        all_images.append(imgs.numpy()); all_labels.append(lbls.numpy().flatten()); all_probs.append(probs)
    all_images   = np.concatenate(all_images).astype("uint8")
    all_labels   = np.concatenate(all_labels).astype(int)
    all_probs    = np.concatenate(all_probs)
    pred_classes = np.argmax(all_probs, axis=1)
    confidences  = np.max(all_probs, axis=1)
    success_idx  = np.where(pred_classes == all_labels)[0]
    failure_idx  = np.where(pred_classes != all_labels)[0]
    sorted_s = success_idx[np.argsort(confidences[success_idx])[::-1]]
    sorted_f = failure_idx[np.argsort(confidences[failure_idx])[::-1]]

    plt.figure(figsize=(16, 8))
    plt.suptitle("VGG19 Top Successes", fontsize=16, weight="bold")
    for i in range(min(6, len(sorted_s))):
        idx = sorted_s[i]; plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
        plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="green", fontsize=9); plt.axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"tl_success_{ts}.png"), dpi=150); plt.show()

    if len(sorted_f) > 0:
        plt.figure(figsize=(16, 8))
        plt.suptitle("VGG19 Failures", fontsize=16, weight="bold")
        for i in range(min(6, len(sorted_f))):
            idx = sorted_f[i]; plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
            plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="red", fontsize=9); plt.axis("off")
        plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"tl_failures_{ts}.png"), dpi=150); plt.show()
    else:
        print("No VGG19 errors detected on this sample.")

    del all_images, all_labels, all_probs, pred_classes, confidences; gc.collect()


execute_tl_error_analysis(
    vgg_model_nw, tl_test_set, TL_CLASS_NAMES, MODEL_SAVE_DIR, f"{TIMESTAMP}_noweight"
)

 
## Partie 3.2 VGG19 avec Class Weights

In [ ]:
import gc
import os
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
DATASET_DIR    = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/datasets/"
MODEL_SAVE_DIR = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/transfer_learning_run"
LOG_DIR        = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/logs/transfer_learning_run"
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 50
SEED = 42
TEST_SPLIT = 0.2
VAL_SPLIT = 0.25
LR_TL      = 0.0001
AUTOUNE    = tf.data.AUTOTUNE
TIMESTAMP  = datetime.now().strftime("%d-%m-%Y-%H-%M-%S")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
def load_tl_datasets_w():
    """Loads three independent splits (60/20/20) for weighted VGG19 training."""
    common_kwargs = dict(
        directory=DATASET_DIR, seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    )
    train_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="training")
    val_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=VAL_SPLIT, subset="validation")
    test_raw = tf.keras.preprocessing.image_dataset_from_directory(**common_kwargs, validation_split=TEST_SPLIT, subset="validation")
    return train_raw, val_raw, test_raw

tl_train_raw_w, tl_val_raw_w, tl_test_raw_w = load_tl_datasets_w()
TL_CLASS_NAMES_W = tl_train_raw_w.class_names
NUM_CLASSES_TL_W = len(TL_CLASS_NAMES_W)

tl_train_set_w = tl_train_raw_w.shuffle(200, seed=SEED).prefetch(AUTOUNE)
tl_val_set_w   = tl_val_raw_w.prefetch(AUTOUNE)
tl_test_set_w  = tl_test_raw_w.prefetch(AUTOUNE)
print(f"VGG19 Weighted: {NUM_CLASSES_TL_W} classes {TL_CLASS_NAMES_W}")

In [ ]:
y_train_tl_w   = np.concatenate([y.numpy().flatten() for _, y in tl_train_set_w])
total_tl       = len(y_train_tl_w)
tl_class_weights = {}
for i in range(NUM_CLASSES_TL_W):
    count = np.sum(y_train_tl_w == i)
    tl_class_weights[i] = total_tl / (NUM_CLASSES_TL_W * count) if count > 0 else 1.0
    print(f"  Class {i} ({TL_CLASS_NAMES_W[i]}): {count} samples | weight = {tl_class_weights[i]:.4f}")

In [ ]:
tl_augmentation_w = Sequential([
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.RandomRotation(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(factor=0.2),
], name="tl_augmentation_weighted")


def build_vgg19_model_w(input_shape, num_classes, augmentation_stack):
    """Instantiates VGG19 backbone (frozen) for the weighted training variant."""
    base_vgg = tf.keras.applications.VGG19(
        include_top=False, weights="imagenet",
        input_shape=input_shape, pooling="avg"
    )
    base_vgg.trainable = False
    model = Sequential([
        augmentation_stack,
        layers.Lambda(
            tf.keras.applications.vgg19.preprocess_input,
            name="vgg19_preprocessing"
        ),
        base_vgg,
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(num_classes),
    ], name="vgg19_weighted_classifier")
    return model


vgg_model_w = build_vgg19_model_w((IMG_HEIGHT, IMG_WIDTH, CHANNELS), NUM_CLASSES_TL_W, tl_augmentation_w)
vgg_model_w.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_TL),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
vgg_model_w.summary()

In [ ]:
tl_log_dir_w    = os.path.join(LOG_DIR, f"{TIMESTAMP}_weighted")
tl_checkpoint_w = os.path.join(MODEL_SAVE_DIR, f"best_vgg19_weighted-{TIMESTAMP}.keras")

tl_callbacks_w = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=tl_checkpoint_w, monitor="val_loss",
        save_best_only=True, mode="min", verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=1, min_lr=1e-7),
    tf.keras.callbacks.TensorBoard(
        log_dir=tl_log_dir_w, histogram_freq=1,
        write_graph=True, update_freq="epoch"
    ),
]

print("Starting VGG19 Weighted Transfer Learning training...")
vgg_history_w = vgg_model_w.fit(
    tl_train_set_w,
    epochs=EPOCHS,
    validation_data=tl_val_set_w,
    callbacks=tl_callbacks_w,
    class_weight=tl_class_weights,
)
print(f"\nTensorBoard logs {tl_log_dir_w}")

In [ ]:
def plot_tl_results_w(history, model, test_dataset, class_names, save_dir, ts, log_dir):
    """Plots VGG19 weighted curves, evaluates on test set, and builds confusion matrix."""
    acc      = history.history["accuracy"]
    val_acc  = history.history["val_accuracy"]
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))
    test_r = model.evaluate(test_dataset, verbose=0)
    print(f"Val Accuracy: {val_acc[-1]*100:.2f}% | Val Loss: {val_loss[-1]:.4f}")
    print(f"→ TEST SET Accuracy: {test_r[1]*100:.2f}% | Loss: {test_r[0]:.4f}")
    os.makedirs(save_dir, exist_ok=True)
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Train", color="royalblue")
    plt.plot(epochs_range, val_acc, label="Val", color="darkorange")
    plt.title("VGG19 Weighted Accuracy"); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Train", color="royalblue")
    plt.plot(epochs_range, val_loss, label="Val", color="darkorange")
    plt.title("VGG19 Weighted Loss"); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"vgg19_w_curves_{ts}.png"), dpi=150); plt.show()
    true_labels, predicted_labels = [], []
    for imgs, lbls in test_dataset:
        logits = model.predict(imgs, verbose=0)
        preds  = np.argmax(tf.nn.softmax(logits).numpy(), axis=1)
        true_labels.extend(lbls.numpy().flatten())
        predicted_labels.extend(preds)
    true_labels = np.array(true_labels); predicted_labels = np.array(predicted_labels)
    cm = confusion_matrix(true_labels, predicted_labels)

    cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix Test Set (VGG19 Weighted)")
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"vgg19_w_cm_{ts}.png"), dpi=150); plt.show()
    print(classification_report(true_labels, predicted_labels, target_names=class_names))
    del true_labels, predicted_labels, cm; gc.collect()


plot_tl_results_w(
    vgg_history_w, vgg_model_w, tl_test_set_w,
    TL_CLASS_NAMES_W, MODEL_SAVE_DIR, f"{TIMESTAMP}_weighted", tl_log_dir_w
)

execute_tl_error_analysis = lambda model, dataset, classes, save_dir, ts: (
    print("Use execute_tl_error_analysis from section 3.1 same logic applies")
)

def execute_tl_error_analysis_w(model, dataset, classes, save_dir, ts):
    """Isolates VGG19 weighted successes and failures."""
    all_images, all_labels, all_probs = [], [], []
    for imgs, lbls in dataset.take(5):
        logits = model.predict(imgs, verbose=0)
        probs  = tf.nn.softmax(logits).numpy()
        all_images.append(imgs.numpy()); all_labels.append(lbls.numpy().flatten()); all_probs.append(probs)
    all_images   = np.concatenate(all_images).astype("uint8")
    all_labels   = np.concatenate(all_labels).astype(int)
    all_probs    = np.concatenate(all_probs)
    pred_classes = np.argmax(all_probs, axis=1)
    confidences  = np.max(all_probs, axis=1)
    success_idx  = np.where(pred_classes == all_labels)[0]
    failure_idx  = np.where(pred_classes != all_labels)[0]
    sorted_s = success_idx[np.argsort(confidences[success_idx])[::-1]]
    sorted_f = failure_idx[np.argsort(confidences[failure_idx])[::-1]]
    plt.figure(figsize=(16, 8))
    plt.suptitle("VGG19 Weighted Top Successes", fontsize=16, weight="bold")
    for i in range(min(6, len(sorted_s))):
        idx = sorted_s[i]; plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
        plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="green", fontsize=9); plt.axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"tl_w_success_{ts}.png"), dpi=150); plt.show()
    if len(sorted_f) > 0:
        plt.figure(figsize=(16, 8))
        plt.suptitle("VGG19 Weighted Failures", fontsize=16, weight="bold")
        for i in range(min(6, len(sorted_f))):
            idx = sorted_f[i]; plt.subplot(2, 3, i + 1); plt.imshow(all_images[idx])
            plt.title(f"True:{classes[all_labels[idx]]}\nPred:{classes[pred_classes[idx]]} ({confidences[idx]*100:.1f}%)", color="red", fontsize=9); plt.axis("off")
        plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"tl_w_failures_{ts}.png"), dpi=150); plt.show()
    del all_images, all_labels, all_probs, pred_classes, confidences; gc.collect()


execute_tl_error_analysis_w(vgg_model_w, tl_test_set_w, TL_CLASS_NAMES_W, MODEL_SAVE_DIR, f"{TIMESTAMP}_weighted")

# Bonus Inférence sur des Images Externes

Cette fonction permet de tester n'importe quel modèle `.keras` (ou `.h5`) sauvegardé sur des images arbitraires. Elle détecte automatiquement si l'architecture est binaire ou multi-classe.

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf


def predict_external_images(model_path, source_path, class_list, img_dims=(128, 128)):
    """Loads a saved Keras model and evaluates it on an external image file or directory."""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found at: {model_path}")
    print(f"Loading model: {os.path.basename(model_path)}")
    model = tf.keras.models.load_model(model_path)
    is_binary = (model.output_shape[-1] == 1)
    print(f"Architecture: {'Binary' if is_binary else 'Multi-Class'} ({model.output_shape[-1]} outputs)")

    source = pathlib.Path(source_path)
    valid_ext = {'.jpg', '.jpeg', '.png', '.webp'}
    image_paths = (
        [source] if source.is_file() and source.suffix.lower() in valid_ext
        else [p for p in source.iterdir() if p.suffix.lower() in valid_ext]
        if source.is_dir() else []
    )
    if not image_paths:
        print(f"No images found at: {source_path}")
        return
    print(f"Found {len(image_paths)} image(s). Running inference...")

    cols = 4
    rows = (len(image_paths) - 1) // cols + 1
    plt.figure(figsize=(16, 4 * rows))
    for i, path in enumerate(image_paths):
        try:
            img_raw    = tf.io.read_file(str(path))
            img_tensor = tf.io.decode_image(img_raw, channels=3, expand_animations=False)
            img_resized = tf.image.resize(img_tensor, img_dims)
            img_batch   = tf.expand_dims(img_resized, axis=0)
            logits      = model.predict(img_batch, verbose=0)
            if is_binary:
                prob       = tf.nn.sigmoid(logits).numpy().flatten()[0]
                pred_idx   = 1 if prob >= 0.5 else 0
                confidence = prob if pred_idx == 1 else (1.0 - prob)
            else:
                probs      = tf.nn.softmax(logits).numpy().flatten()
                pred_idx   = np.argmax(probs)
                confidence = probs[pred_idx]
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img_resized.numpy().astype("uint8"))
            color = "green" if pred_idx == (1 if is_binary else 0) else "steelblue"
            plt.title(f"{path.name}\n{class_list[pred_idx]} ({confidence*100:.1f}%)",
                      color=color, fontsize=9)
            plt.axis("off")
        except Exception as err:
            print(f"Could not process {path.name}: {err}")
    plt.tight_layout()
    plt.show()


# Configuration adapt paths to your environment
MY_MODEL       = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/binary_model_run/best_binary_noweight-XX-XX-XX.keras"
MY_MODEL_MULTI = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/multi_class_model_run/best_mc_noweight-XX-XX-XX.keras"
MY_MODEL_TL    = "/home/polo/Backup_WSL/dev/ia-opt-cesi/projet/saved_models/transfer_learning_run/best_vgg19_noweight-XX-XX-XX.keras"
TEST_FOLDER    = "/home/polo/Backup_WSL/test_images/"

# Uncomment the inference you want to run:
# predict_external_images(MY_MODEL,       TEST_FOLDER, ["Not-Photo", "Photo"])
# predict_external_images(MY_MODEL_MULTI, TEST_FOLDER, ["Class1", "Class2", "Class3", "Class4", "Class5"])  # adapt class list
# predict_external_images(MY_MODEL_TL,    TEST_FOLDER, ["Class1", "Class2", "Class3", "Class4", "Class5"])

# Conclusion
